# Which new study programme for the EMI faculty? Progress notebook

**Question.** Which new degree programme would bring the most *new* students to OTH Amberg-Weiden and can be taught by the EMI faculty with the staff and labs it already has?

**How we work.** The programme has to come out of the data. We do not start from programmes we picked ourselves. The work runs in nine steps, one at a time. Every step has rules that were written down before the step ran.

This notebook grows by one section per finished step, so it always shows where the project stands.

- Rules: `configs/funnel.yaml`
- Why each rule is there: `docs/funnel_preregistration.md`
- Decisions with dates: `docs/decision_log.md`

In [1]:
import os, sys, re, yaml, pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 200)

def show(df):
    display(df.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "normal"})
            .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))

RULES_TEXT = (ROOT / "configs" / "funnel.yaml").read_text(encoding="utf-8")
RULES = yaml.safe_load(RULES_TEXT)
print("Rules loaded:", len(RULES), "blocks -", ", ".join(RULES))

Rules loaded: 17 blocks - definitions, universe, text, similarity, discovery, pool_mapping, screen, freeze, demand, labour, capture, shortlist, feasibility_gate, ranking, stress_tests, decision, conformance


## Where we are

In [2]:
STATUS = [   # updated after every finished step
    (0, "Rules", "Which rules decide?", "done"),
    (1, "Data inventory", "Is the data there and current?", "done"),
    (2, "Programme types", "What kinds of programme exist in Germany?", "done (types checked again in Step 3)"),
    (3, "Screen", "Which types are open to OTH and in EMI's field? How much competition is near?", "done (competition counts not validated)"),
    (4, "Student trend", "Are students coming?", "done"),
    (5, "Job ads", "Are there jobs nearby?", "done"),
    (6, "New students", "How many first-years would be new to OTH?", "next"),
    (7, "Feasibility", "Can EMI teach it with what it has?", "open"),
    (8, "Decision", "Which programme, and how sure are we?", "open"),
]
show(pd.DataFrame(STATUS, columns=["Step", "Name", "Question", "Status"]))

Step,Name,Question,Status
0,Rules,Which rules decide?,done
1,Data inventory,Is the data there and current?,done
2,Programme types,What kinds of programme exist in Germany?,done (types checked again in Step 3)
3,Screen,Which types are open to OTH and in EMI's field? How much competition is near?,done (competition counts not validated)
4,Student trend,Are students coming?,done
5,Job ads,Are there jobs nearby?,done
6,New students,How many first-years would be new to OTH?,next
7,Feasibility,Can EMI teach it with what it has?,open
8,Decision,"Which programme, and how sure are we?",open


## Why we started again

- Our first analysis started from programmes **we** had chosen (cybersecurity, robotics). That is a biased start: the result can only be one of our own ideas.
- A first data-driven run was reviewed and is **not** used as a result. The code had applied several rules differently from how they were written, and one test (competition) was not valid.
- So the analysis is redone step by step. The rules come first, and each step is checked before the next one starts.

## What the supervisor told us (meeting and reply email)

| The supervisor said | What we do with it |
|---|---|
| An entire new degree programme, not a few extra modules. It must not already exist at OTH. | Only whole programmes are candidates. Anything OTH already offers is removed. |
| Bachelor or master, both are fine. | Both levels are analysed. |
| An overlap of 40-80 % with existing lectures is ideal. Too much overlap is just a relabelled programme. Too little costs too many resources. | At least 40 % of the credits must be covered by modules EMI already teaches. At most 80 % may come from one existing programme. |
| He has no minimum number of students. We should assume students come and **argue why**. | We keep 20 first-years as our own assumption and show what changes with 15, 25 or no minimum. |
| It need not be unique in Germany. Programmes abroad are fine as inspiration. | Competition is measured and used as an argument. It does not remove a candidate by itself. |
| Good arguments: it does not exist in Germany, or a few similar programmes elsewhere (for example in the north) attract many students and there is nothing comparable in the south. | Every candidate gets a north/south check: does it exist elsewhere, is it missing in Bavaria and the south, and how well can we show it is popular there. |
| He cannot give student numbers per course. He gave us the lecture timetables of the faculty. | The timetables show which topics current lecturers already teach. They stay outside the repository. |
| Sources: other universities and the Federal Statistical Office. | Unchanged: Destatis, the national study catalogue, other universities' module handbooks, the job board of the employment agency. |
| What counts is the chain of reasoning and the documentation. | Every step ends with short notes on why each programme stays or goes. |

## Step 0: the rules, fixed before any data is touched

**Question.** Which rules decide, so that nobody (including us) can bend them later to get a favourite result?

**What we did.** We wrote every rule and every threshold into one file and committed it before running any analysis. The numbers below are read from that file.

In [3]:
g, c, s, d = RULES["feasibility_gate"], RULES["capture"], RULES["screen"], RULES["decision"]["rule"]
rows = [
    (2, "Programme types", "All full-time bachelor and master offers in Germany are grouped by their text. No topic list is written by hand.",
        f"a group is stable from an agreement of {RULES['discovery']['stable_jaccard']} across repeated runs"),
    (3, "Already at OTH", "A type is removed if an OTH programme of that level sits in its centre.", "closer than half of the type's own programmes"),
    (3, "Teacher training", "Removed: a faculty cannot open a state teaching degree on its own.", f"from a share of {s['teacher_training']['drop_if_at_least']:.0%}"),
    (3, "Shrinking field", "Removed if German first-years in Bavaria fall and the trend has predicted well in the past.", f"below {s['trend']['drop_if_below']} % a year"),
    (3, "Outside EMI", "Removed if the content is far from what EMI teaches.", "bar set by a calibration on OTH's own programmes"),
    (3, "Competition", "Counted by content similarity, whatever the programme is called. Not a cut.", "within 50, 100 and 150 km of Amberg"),
    (3, "North/south gap", "Does it exist elsewhere but not in Bavaria or the south?", "flags and a Bavarian quotient"),
    (5, "Job ads", "Keywords come from the type's own programme titles, never typed by hand.", f"counted in {len(RULES['labour']['regions'])} regions"),
    (6, "Minimum cohort", "Our own assumption; the supervisor gave none.", f"{c['min_cohort']['value']} first-years (central estimate)"),
    (7, "Overlap, lower end", "Share of credits covered by modules EMI already teaches.", f"at least {g['band']['min']:.0%}"),
    (7, "Overlap, upper end", "Share of credits that come from one existing OTH programme.", f"at most {g['one_programme']['max']:.0%}"),
    (7, "Existing facilities only", "New modules must be on topics a current lecturer already teaches. No new lab.", "checked by hand on two real curricula"),
    (8, "Decision", "Recommend the top candidate only if it is clearly ahead.", d["two_or_more"].replace("recommend the top one if ", "")),
]
show(pd.DataFrame(rows, columns=["Step", "Rule", "In plain words", "Number"]))

Step,Rule,In plain words,Number
2,Programme types,All full-time bachelor and master offers in Germany are grouped by their text. No topic list is written by hand.,a group is stable from an agreement of 0.75 across repeated runs
3,Already at OTH,A type is removed if an OTH programme of that level sits in its centre.,closer than half of the type's own programmes
3,Teacher training,Removed: a faculty cannot open a state teaching degree on its own.,from a share of 50%
3,Shrinking field,Removed if German first-years in Bavaria fall and the trend has predicted well in the past.,below -3.0 % a year
3,Outside EMI,Removed if the content is far from what EMI teaches.,bar set by a calibration on OTH's own programmes
3,Competition,"Counted by content similarity, whatever the programme is called. Not a cut.","within 50, 100 and 150 km of Amberg"
3,North/south gap,Does it exist elsewhere but not in Bavaria or the south?,flags and a Bavarian quotient
5,Job ads,"Keywords come from the type's own programme titles, never typed by hand.",counted in 3 regions
6,Minimum cohort,Our own assumption; the supervisor gave none.,20 first-years (central estimate)
7,"Overlap, lower end",Share of credits covered by modules EMI already teaches.,at least 40%


### The feasibility rule in one picture

```
overlap with ALL modules EMI already teaches
0 % ............ 40 % ...................................... 100 %
   too little            enough existing lectures to reuse
   (costs too many
    resources)

share that comes from ONE existing OTH programme
0 % ...................................... 80 % ............ 100 %
   a new programme                              a relabelled copy
                                                of that programme
```

A candidate passes if it is right of the 40 % line in the first bar and left of the 80 % line in the second. The recommendation is then taken only from candidates that existing staff and labs can carry.

In [4]:
# Where does every number in the rules come from? Each one carries a tag in the rules file.
TAGS = {"S": "the supervisor", "L": "literature or regulation", "C": "carried over from earlier pre-registered rules",
        "J": "our own judgement, tested with other values", "K": "written knowing a number of the first run (declared)",
        "N": "size or pass rule of a manual check"}
body = RULES_TEXT.split("definitions:", 1)[1]
counts = {t: len(re.findall(rf"\[{t}\]", body)) for t in TAGS}
show(pd.DataFrame([(f"[{t}]", TAGS[t], n) for t, n in counts.items()], columns=["Tag", "Origin of the number", "Times used"]))

Tag,Origin of the number,Times used
[S],the supervisor,8
[L],literature or regulation,3
[C],carried over from earlier pre-registered rules,38
[J],"our own judgement, tested with other values",17
[K],written knowing a number of the first run (declared),14
[N],size or pass rule of a manual check,14


In [5]:
# The stress tests: every judgement number is run with other values, to see whether the answer changes.
st = RULES["stress_tests"]
rows = [("after the candidate list is fixed", k.replace("_", " "), "; ".join(map(str, v)) if isinstance(v, list) else str(v)) for k, v in st["downstream_factorial"].items()]
rows += [("changes the candidate list", k.replace("_", " "), "; ".join(map(str, v)) if isinstance(v, list) else str(v)) for k, v in st["list_changing_one_at_a_time"].items()]
print(len(rows), "stress settings")
show(pd.DataFrame(rows, columns=["Kind", "What is varied", "Values"]))

32 stress settings


Kind,What is varied,Values
after the candidate list is fixed,min cohort,20; 25
after the candidate list is fixed,gross band,"[0.33, 3.0]; [0.5, 2.0]"
after the candidate list is fixed,pool trend,unprojected; projected; foreign added flat
after the candidate list is fixed,master transition,0.5; 0.3; 0.7
after the candidate list is fixed,dirichlet concentration,10; 50; 200
after the candidate list is fixed,overlap band,"[0.4, 0.8]; [0.5, 0.8]; [0.3, 0.9]"
after the candidate list is fixed,overlap reading,lower end on all EMI modules and upper end on one programme; both ends on all EMI modules
after the candidate list is fixed,one programme max,0.8; 0.5; none
after the candidate list is fixed,matching threshold,0.463; 0.285
after the candidate list is fixed,existing resources N cap,none; 20 bachelor and 10 master; 10 bachelor and 5 master


### How the rules were checked

An independent reviewer, who had not seen our reasoning or any earlier result, attacked the rules three times.

| Round | Blocking problems found | Other major points | What happened |
|---|---|---|---|
| 1 | 11 | 10 | rules rewritten |
| 2 | 9 | 5 | rules reworded |
| 3 | 3 | 0 | reworded as proposed; not reviewed a fourth time |

| # | Criterion (written before the rules) | Target | Result | Pass |
|---|---|---|---|---|
| C1 | Every problem of the first run has a rule that prevents it | 14 of 14 | 14 of 14 | yes |
| C2 | Every point of the supervisor is in the rules, faithfully | 9 of 9 | 9 of 9 | yes |
| C3 | Every own-judgement number has a stress value | all | a listed group has none (mostly sizes of manual checks) | **no** |
| C4 | Every rule names its data source or a fallback | none missing | none missing | yes |
| C5 | No result produced or read in this step | none | only rule and document files changed | yes |
| C6 | No rule that two people could apply differently | 0 blockers | 0 after round 3 (last 3 fixes not re-reviewed) | yes |
| C7 | What we already knew is declared | complete | 12 rules marked as written knowing a first-run number | yes |
| C8 | No lecturer name in the repository | 0 | 0 | yes |
| C9 | All tests pass | all | 116 pass, 1 older test fails (not from this step) | **no** |

The two failed criteria were reported and accepted as exceptions; they are listed in `docs/funnel_preregistration.md`.

### What we achieved in Step 0

We now have one fixed rulebook for the whole analysis, written before any data is touched. It holds the supervisor's answers, a fix for every mistake of the first run, and a measure for his tip "popular elsewhere, missing in the south". Because the rules came first, the result cannot be steered towards a favourite programme.

**Strongest objection.** Some numbers are still our own judgement (for example the minimum of 20 students). **Answer.** Each of them is marked as such and is run with other values in Step 8, so we will see whether the recommendation depends on it.

## Step 1: data inventory

**Question.** Does every rule have the data it needs, how old is the data, and what is missing?

**What we did.**
- Listed every data source with its size, period, date and gaps.
- Downloaded the one public dataset that was missing: first-year students by subject for the 15 other federal states (official Destatis interface). The north/south check needs it.
- Read the two timetable PDFs from the supervisor into one table. Lecturers appear only as labels (L001, L002, ...).
- Fixed one number that Step 7 needs (the matching threshold) before any programme is looked at.

No programme was grouped or scored in this step.

In [6]:
import json
INV = json.loads((ROOT / "data" / "interim" / "funnel2_inventory.json").read_text(encoding="utf-8"))
cat, emi, tt, oth, thr = INV["catalogue"], INV["emi"], INV["timetables"], INV["other"], INV["threshold"]
by = {t["table"]: t for t in INV["destatis"]}
laender_ok = sum(v["covers_2019_2024"] for v in INV["laender"].values())
print("Inventory run on", INV["stamp"]["date"], "with code version", INV["stamp"]["commit"])

rows = [
    ("Study catalogue of Germany", f"{cat['rows']:,} offers; {cat['after_dedupe']:,} bachelor and master offers after merging variants; {cat['full_time_after_dedupe']:,} full-time",
        "19 Oct 2026", f"{cat['offers_empty_after_drops']:,} offers have no content text once filler is removed; no language field"),
    ("EMI modules (OTH handbooks)", f"{emi['modules']} modules, {emi['by_level']['BA']} bachelor and {emi['by_level']['MA']} master", "19 Oct 2026", f"{emi['ects_missing']} modules without credits"),
    ("First-years and graduates by subject, Bavaria and Germany", f"{by['21311-0015_BY']['subjects']} subjects", f"{by['21311-0015_BY']['first_year']}-{by['21311-0015_BY']['last_year']}", "none"),
    ("First-years by subject, the 15 other federal states (new)", f"{laender_ok} of {len(INV['laender'])} states", "2018-2025", "none"),
    ("Job ads", "; ".join(f"{d}: {n}" for d, n in oth["job_snapshots"].items()) + " counts", "Oct-Nov 2026", "new keywords have no history"),
    ("School leavers by county", f"{oth['school_leavers']['counties_last_year']} counties", oth["school_leavers"]["years"], "year " + ", ".join(map(str, oth["school_leavers"]["missing_years"])) + " missing"),
    ("Students by campus (for master programmes)", f"{oth['master_origins']['campuses']} campuses, {oth['master_origins']['students']:,} students", "winter 2024/25", "none"),
    ("EMI timetables (internal, from the supervisor)", f"{tt['groups']} group plans, {tt['grid_cells']:,} timetable entries, {tt['distinct_subject_names']} subjects, {tt['lecturers']} lecturers",
        ", ".join(tt["semesters"]), "no credits, no hours, no student numbers"),
]
show(pd.DataFrame(rows, columns=["Data source", "Size", "Period or date", "Gaps"]))

Inventory run on 2026-11-14 with code version 8fddcb2


Data source,Size,Period or date,Gaps
Study catalogue of Germany,"22,093 offers; 19,872 bachelor and master offers after merging variants; 14,902 full-time",19 Oct 2026,"10,139 offers have no content text once filler is removed; no language field"
EMI modules (OTH handbooks),"447 modules, 333 bachelor and 114 master",19 Oct 2026,38 modules without credits
"First-years and graduates by subject, Bavaria and Germany",297 subjects,2005-2025,none
"First-years by subject, the 15 other federal states (new)",15 of 15 states,2018-2025,none
Job ads,2026-10-15: 48; 2026-10-17: 48; 2026-11-07: 92 counts,Oct-Nov 2026,new keywords have no history
School leavers by county,96 counties,2014-2024,year 2019 missing
Students by campus (for master programmes),"17 campuses, 83,481 students",winter 2024/25,none
"EMI timetables (internal, from the supervisor)","80 group plans, 1,028 timetable entries, 314 subjects, 98 lecturers","SS 2026, WS 2026/27","no credits, no hours, no student numbers"


### A closer look at the study catalogue

This is the list of all programmes in Germany that Step 2 will group. About half of the offers describe their content only with a standard sentence ("please ask the university"). For those, only the title and the subject labels can be used.

In [7]:
n = cat["after_dedupe"]
rows = [("Bachelor offers", cat["bachelor_after_dedupe"]), ("Master offers", cat["master_after_dedupe"]),
        ("Full-time offers (used to find the programme types)", cat["full_time_after_dedupe"]),
        ("Offers with focus areas listed", round(cat["share_with_focus_items"] * n)),
        ("Offers with no content text once the standard sentence and the university's note are removed", cat["offers_empty_after_drops"]),
        ("Offers of OTH Amberg-Weiden", cat["oth_offers"]), ("Institutions", cat["institutions"])]
t = pd.DataFrame(rows, columns=["", "Number"])
t["Share of all offers"] = [f"{v / n:.0%}" if i < 6 else "" for i, v in enumerate(t["Number"])]
show(t)
show(pd.DataFrame(sorted(cat["institution_types"].items(), key=lambda kv: -kv[1]), columns=["Type of institution", "Offers"]))

,Number,Share of all offers
Bachelor offers,9703,49%
Master offers,10169,51%
Full-time offers (used to find the programme types),14902,75%
Offers with focus areas listed,9499,48%
Offers with no content text once the standard sentence and the university's note are removed,10139,51%
Offers of OTH Amberg-Weiden,46,0%
Institutions,404,


Type of institution,Offers
Universität,11699
Fachhochschule/Hochschule für angewandte Wissenschaften,6738
Kunst- und Musikhochschule,986
Hochschule eigenen Typs,276
Berufsakademie/Duale Hochschule,124
Verwaltungshochschule,29
Private Hochschule,20


### The timetables: who teaches what

One row per timetable entry. The lecturer column holds a neutral label; the names stay outside the repository. Four pages were compared with the PDFs by eye: 59 of 59 entries match. A second reading of all pages with another tool gave the same 1,028 entries.

In [8]:
tts = pd.read_csv(ROOT / "data" / "interim" / "timetable_subjects.csv")
grid = tts[tts["in_grid"]]
show(pd.DataFrame([(f["semester"], f["pages"], f["groups"], f["grid_cells"], f["listed_subjects_without_a_slot"], f["cell_texts_in_no_cell"]) for f in tt["files"]],
                  columns=["Semester", "Pages read", "Group plans", "Timetable entries", "Subjects without a weekly slot", "Texts left over"]))
sample = grid[(grid["semester"] == "WS 2026/27") & (grid["group"] == "KI 1")][["group", "day", "subject", "lecturer", "room"]].head(8)
show(sample.fillna(""))
print("Subjects taught per lecturer (median):", grid.groupby("lecturer")["subject"].nunique().median())
print("Distinct sessions (a session shared by several groups counted once):", tt["distinct_sessions"])
print("EMI handbook module titles also found as a timetable subject (same title, small spelling differences ignored):", tt["emi_distinct_titles_found_normalised"], "of", tt["emi_distinct_titles"])

Semester,Pages read,Group plans,Timetable entries,Subjects without a weekly slot,Texts left over
SS 2026,80,40,553,10,0
WS 2026/27,80,40,475,9,0


group,day,subject,lecturer,room
KI 1,Mo,Mathematik 1 (Zusatzübung),L029,EMI313
KI 1,Mo,Robotics Starter,L056,DC107
KI 1,Di,Mathematik 1 (Vorlesung),L029,EMI301
KI 1,Di,Robotics Starter,L056,DC107
KI 1,Di,Betriebswirtschaftslehre & Innovationsmanagement,L079,EMI214
KI 1,Mi,Mathematik 1 (Übung),L029,EMI002
KI 1,Mi,Symbolische Künstliche Intelligenz 1,L077,EMI207
KI 1,Mi,Symbolische Künstliche Intelligenz 1,L056,EMI207


Subjects taught per lecturer (median): 2.0
Distinct sessions (a session shared by several groups counted once): 494
EMI handbook module titles also found as a timetable subject (same title, small spelling differences ignored): 169 of 293


### The matching threshold for Step 7

In Step 7 a computer compares each module of a reference curriculum with EMI's modules. Two modules count as "the same topic" above a similarity threshold. The threshold was fixed now, before any candidate exists, with the procedure the project already used.

In [9]:
show(pd.DataFrame([("Threshold (written into the rules)", thr["value"]), ("Value in the earlier feasibility report", thr["earlier_report_value"]),
                   ("Correct matches found", f"{thr['tpr']:.0%}"), ("False matches accepted", f"{thr['fpr']:.0%}"),
                   ("Reference modules used", f"{thr['positives']} with a match, {thr['negatives']} without, from {len(thr['calibration_reference_programmes'])} programmes of the earlier analysis")], columns=["", "Value"]))
print("The automatic match is only moderately reliable, so every match is checked by hand in Step 7.")

,Value
Threshold (written into the rules),0.285000
Value in the earlier feasibility report,0.285000
Correct matches found,66%
False matches accepted,14%
Reference modules used,"80 with a match, 88 without, from 7 programmes of the earlier analysis"


The automatic match is only moderately reliable, so every match is checked by hand in Step 7.


### What we achieved in Step 1

We now know exactly what data we have and what we do not have. Everything the rules need is there. One missing piece was downloaded: student numbers by subject for all other federal states, so we can later test the supervisor's idea "popular elsewhere, missing in the south". The timetables are readable as a table without any names. All 125 automatic tests pass.

**Limits found.** Half of the catalogue has no content text. The catalogue has no field for the teaching language. The timetables show who teaches what, but not how many hours anyone has free.

## Step 1b: removing a bias from the matching threshold

**The problem.** In Step 7 the computer decides whether an existing EMI module covers a module of another university's curriculum. It does so with a similarity threshold. The threshold we had (0.285) was tuned on cybersecurity and robotics curricula, the two candidates of our earlier analysis. A tool tuned on the old favourites could favour them again.

**What we did.** We tuned the threshold again using only EMI's own module handbooks:
- A module that appears under the same title in two different handbooks, with independently written text, is a *true match*.
- The most similar module with a completely different title is the *hardest wrong match*.
- The procedure was written down and committed before the number was computed, and the code ran once.

In [10]:
THR = json.loads((ROOT / "data" / "interim" / "funnel2_threshold.json").read_text(encoding="utf-8"))
show(pd.DataFrame([
    ("Data used", f"EMI's own handbooks only: {THR['modules_read']} modules in {THR['handbook_files']} handbook files"),
    ("Same-title pairs in two handbooks", f"{THR['same_title_pairs']}, of which {THR['copies_left_out']} are copied texts and left out"),
    ("Modules used for the test", f"{THR['anchors']}"),
    ("New threshold", f"{THR['value']}  (plausible range {THR['bootstrap_interval'][0]} to {THR['bootstrap_interval'][1]})"),
    ("True matches found at the new threshold", f"{THR['tpr']:.0%}"),
    ("Wrong matches accepted at the new threshold", f"{THR['fpr']:.0%}"),
    ("Quality score (AUC; 0.5 = chance, 1.0 = perfect)", f"{THR['auc']}" + ("  -> weak" if THR["weak"] else "")),
    ("The old threshold 0.285 on this test", f"finds {THR['former_value_on_this_set']['tpr']:.0%} of true matches but also accepts {THR['former_value_on_this_set']['fpr']:.0%} of wrong ones"),
], columns=["", "Result"]))

,Result
Data used,EMI's own handbooks only: 447 modules in 13 handbook files
Same-title pairs in two handbooks,"190, of which 74 are copied texts and left out"
Modules used for the test,121
New threshold,0.463 (plausible range 0.382 to 0.538)
True matches found at the new threshold,57%
Wrong matches accepted at the new threshold,28%
"Quality score (AUC; 0.5 = chance, 1.0 = perfect)",0.627 -> weak
The old threshold 0.285 on this test,finds 83% of true matches but also accepts 82% of wrong ones


### What the result means

A second look after the run showed two things:

| | |
|---|---|
| **The computer is good at ranking.** | For 74 % of the test modules the right partner is among the 3 most similar of about 408 modules; for 51 % it is number 1. |
| **No single cut-off is reliable.** | Any threshold between about 0.38 and 0.47 is nearly as good as 0.463. And two handbooks of the same university share wording, so the value is too strict for comparing with another university. |

So neither the old 0.285 nor the new 0.463 should decide anything on its own.

### What we achieved in Step 1b

We removed the tuning on the old favourite programmes and learned something more useful: the computer can reliably *suggest* which EMI modules are close to a given module, but it cannot reliably *decide* whether they match. That decision should be made by a person.

**Decision (Soma).** No threshold decides a module match. In Step 7 the computer lists the three closest EMI modules for every module of a reference curriculum. An independent AI reviewer reads the texts and marks every row by rules written beforehand. Soma checks 10 rows per candidate and every row the reviewer is unsure about; if 2 or more of the 10 are wrong, Soma marks all rows of that candidate. The second team member repeats one candidate per level without seeing any marks.

## Step 2: programme types, found from the data

**Question.** What kinds of programme exist in Germany? In our first analysis we wrote the topic lists ourselves (cybersecurity, robotics, ...). Now the data has to show the kinds by itself.

**What we did.**
1. For every offer we built one short text: the title without degree words, its focus areas, and its subject labels. The standard sentence "ask the university for the contents" and the universities' notes were removed.
2. A language model turned every text into numbers, so that offers with similar content lie close together.
3. A grouping method looked for dense groups among the full-time offers, with 10 different settings. A group counts as *stable* only if it comes back when the grouping is repeated on changing 80 % samples.
4. The offers left over were grouped again. A group at 2 or more universities becomes a *rare type* (fewer than 10 offers) or a *mid-sized type* (10 or more). Rare programmes are welcome under our scope, so they must not get lost.
5. Offers of other study forms, and full-time offers still left over, were added afterwards to the nearest type, if they are close enough.

No type is judged in this step. Nothing about OTH or EMI is used.

### The first result did not pass our own quality check, so we changed three rules

Before the step we had fixed a bar: an independent reader looks at 30 types drawn at random, 10 titles each. A type passes if at least 8 of 10 titles are the same kind of programme. **At least 24 of the 30 types must pass.**

Our first rule picked the coarsest setting. Its types were whole fields that mix neighbouring subjects (geography with geophysics and urban planning, political science with sociology), and 30 mid-sized groups such as Architektur, Psychologie and Medieninformatik were dropped. We tested finer settings and the catalogue's own subject labels, each with a different independent reader.

Soma then decided: (1) use the finest setting, (2) keep the mid-sized groups, with a flag, (3) in Step 3, read every type that survives the screen; a mixed type is flagged and cannot become a candidate without a full reading of its titles. The bar itself was not lowered.

In [11]:
CH = json.loads((ROOT / "data" / "interim" / "funnel2_step2_checks.json").read_text(encoding="utf-8"))
n = CH["types_per_sample"]
show(pd.DataFrame([(r["grouping"], r["types"], f"{r['first']} of {n} ({r['first'] / n:.0%})", f"{r['second']} of {n} ({r['second'] / n:.0%})", r["note"]) for r in CH["rows"]],
                  columns=["Grouping", "Types", "First sample: pass", "Second sample: pass", ""]))
print(f"Bar: {int(CH['bar'] * n)} of {n}. The redo, first sample by kind of type:", CH["redo_by_kind"])

Grouping,Types,First sample: pass,Second sample: pass,
First result (our first rule: smallest group 50),"63 stable, 229 rare",17 of 30 (57%),14 of 30 (47%),replaced
Test: smallest group 20,"140 stable, 425 rare",19 of 30 (63%),20 of 30 (67%),test only
Test: smallest group 10 (finest),"263 stable, 409 rare",21 of 30 (70%),21 of 30 (70%),test only
Test: the catalogue's own subject labels,339 labels,10 of 30 (33%),13 of 30 (43%),test only
"Redo (changed rules: finest setting, mid-sized groups kept)","233 stable, 130 mid-sized, 461 rare",22 of 30 (73%),22 of 30 (73%),our result


Bar: 24 of 30. The redo, first sample by kind of type: {'stable': '5 of 6', 'mid-sized': '4 of 4', 'rare': '13 of 20'}


### The result we carry forward (the redo)

In [12]:
S2 = json.loads((ROOT / "data" / "interim" / "funnel2_step2.json").read_text(encoding="utf-8"))
E2 = json.loads((ROOT / "data" / "interim" / "funnel2_embed.json").read_text(encoding="utf-8"))
sel, ft, lo = S2["selected"], S2["full_time"], S2["left_over"]
show(pd.DataFrame([
    ("Offers with a text", f"{S2['offers']:,} (bachelor and master, Germany)"),
    ("Offers without focus areas (title and subject labels only)", f"{E2['without_focus_items']:,}"),
    ("Full-time offers used to find the types", f"{ft:,}"),
    ("Setting", f"smallest group {sel['min_cluster_size']} offers (our first rule would pick {S2['former_rule_setting'][0]})"),
    ("Stable types (come back when the grouping is repeated)", S2["stable_types"]),
    ("Mid-sized types (10 or more offers; not shown to be stable)", S2["mid_sized_types"]),
    ("Rare types (2 to 9 offers at 2 or more universities; not stability-tested)", S2["rare_types"]),
    ("Full-time offers in a stable type", f"{S2['full_time_in_stable']:,} ({S2['full_time_in_stable'] / ft:.0%})"),
    ("Full-time offers in a mid-sized type", f"{S2['full_time_in_mid_sized']:,} ({S2['full_time_in_mid_sized'] / ft:.0%})"),
    ("Full-time offers in a rare type", f"{S2['full_time_in_rare']:,} ({S2['full_time_in_rare'] / ft:.0%})"),
    ("Full-time offers without a type (single offers, or groups at one university only)", f"{S2['full_time_without_type']:,} ({S2['full_time_without_type'] / ft:.0%})"),
    ("Offers added afterwards to the nearest type", f"{S2['assigned_afterwards']:,}"),
    ("Offers without any type at the end", f"{S2['offers_without_any_type']:,} ({S2['offers_without_any_type'] / S2['offers']:.0%})"),
    ("Same result when the grouping is run a second time", CH["reproduced"]),
], columns=["", "Result"]))

,Result
Offers with a text,"19,872 (bachelor and master, Germany)"
Offers without focus areas (title and subject labels only),"10,416"
Full-time offers used to find the types,"14,902"
Setting,smallest group 10 offers (our first rule would pick 50)
Stable types (come back when the grouping is repeated),233
Mid-sized types (10 or more offers; not shown to be stable),130
Rare types (2 to 9 offers at 2 or more universities; not stability-tested),461
Full-time offers in a stable type,"8,063 (54%)"
Full-time offers in a mid-sized type,"3,702 (25%)"
Full-time offers in a rare type,"1,814 (12%)"


### What a type looks like

The titles are the offers closest to the centre of the type. First the 12 largest types, then 12 types drawn at random.

In [13]:
TY = pd.read_csv(ROOT / "data" / "interim" / "funnel2_types.csv")
cols = {"type": "Type", "kind": "Kind", "central titles": "Titles closest to the centre", "members": "Full-time offers", "assigned offers": "All offers"}
show(TY.sort_values("members", ascending=False).head(12)[list(cols)].rename(columns=cols))
show(TY.sample(12, random_state=1).sort_values("type")[list(cols)].rename(columns=cols))

Type,Kind,Titles closest to the centre,Full-time offers,All offers
34,stable,Erziehungswissenschaften/Pädagogik; Erziehungs- und Bildungswissenschaften; Erziehungswissenschaften; Erziehungswissenschaft; Pädagogik/Bildungswissenschaft,186,259
185,stable,Wirtschaftsingenieurwesen; Wirtschaftsingenieurwesen - Industrie; Wirtschaftsingenieurwesen / Maschinenbau; Wirtschaftsingenieurwesen Maschinenbau; Wirtschaftsingenieurwesen Elektrotechnik,166,215
216,stable,Maschinenbau; Maschinenbau und Design; Maschinenbau/Verfahrenstechnik; Allgemeiner Maschinenbau; Maschinenbau und Verfahrenstechnik,163,201
243,mid-sized,Betriebswirtschaftslehre; Betriebswirtschaft; Business Administration; Business Management; Betriebswirtschaftslehre (BWL),152,206
84,stable,Wirtschaftsinformatik; Wirtschaft und Informatik; Wirtschaftsinformatik/ Business Intelligence; Wirtschaftswissenschaftliche Informatik; Wirtschaftsinformatik - Digital Business & Analytics,144,183
254,mid-sized,Sonderpädagogik Lehramt Sonderpädagogik; Lehramt Sonderpädagogik Lehramt Sonderpädagogik; Lernen Sonderpädagogik; Deutsch Sonderpädagogik; Lernen Lehramt Sonderpädagogik,138,158
193,stable,Bauingenieurwesen; Bauingenieurwesen und Umweltingenieurwesen; Bauingenieurwesen und Umweltwissenschaften; Wirtschaftsingenieurwesen / Bauingenieurwesen; Bau- und Umweltingenieurwesen,131,161
262,mid-sized,Betriebswirtschaft; Betriebswirtschaftslehre; Betriebswirtschaft & Management; Business Management; Business Administration,125,169
331,mid-sized,Elektrotechnik; Elektrotechnik und Informationstechnik; Elektro- und Informationstechnik; Elektrotechnik/ Informationstechnik; Informations- und Elektrotechnik,125,147
76,stable,English and American Studies; Anglistik/Amerikanistik; English and American Studies / English Linguistics; English and American Studies/Anglistik und Amerikanistik; Anglistik / Amerikanistik,104,146


Type,Kind,Titles closest to the centre,Full-time offers,All offers
8,stable,Chemie; Chemistry; Chemie - dual; Business Chemistry; Chemistry of Functional Molecules and Materials,37,56
117,stable,Textil- und Bekleidungstechnik; Textile Produkte; Bekleidung - Technik und Management; Textiles Ingenieurwesen; Textile and Clothing Management,16,19
239,mid-sized,Biochemie; Biologie; Chemie; Chemische Biotechnologie; Medizinische Biologie,27,28
274,mid-sized,Medizintechnik; Biomedical Engineering; Biomedizinische Technik; Medizintechnik/Medical Engineering; Medizintechnik/Biomedical Engineering,83,96
353,mid-sized,Sachunterricht (Lehramt an Grundschulen); Sachunterricht Grundschullehramt; Sachunterricht; Sachunterricht (Lehramt an Grundschulen) (Lehramt an Grundschulen); Sachunterricht Lehramt an Grundschulen,12,15
415,rare,Insolvenzrecht und Reorganisationsverfahren; Unternehmensrestrukturierung und Insolvenzmanagement,2,2
447,rare,Deutsch-Französischer Rechtswissenschaft: Economics and Institutions; Deutsch-Franzöischer Nationales und Europäisches Wirtschaftsrecht Law; Deutsches und Französisches Recht,3,3
504,rare,BWL techn. (Technisch orientierte Betriebswirtschaftslehre); Betriebswirtschaftliche Forschung,3,3
598,rare,Asian Studies and International Management Studienrichtung China / Southeast Asia & India; International Business Management East Asia,2,2
629,rare,"Filmkultur: Archivierung, Programmierung, Präsentation; Filmkulturerbe",2,2


### What the result means

- **The types are finer and cleaner than in the first result:** 22 of 30 hold one kind of programme (before: 17 of 30). This is still below our bar of 24, and we say so. About 1 type in 4 is mixed.
- **That is why Step 3 reads every type that survives the screen.** A mixed type is flagged and cannot become a candidate without a full reading of its titles.
- **Only the stable types were shown to be stable.** Mid-sized and rare types carry a flag. Mid-sized types hold mainstream kinds such as Betriebswirtschaft, Elektrotechnik, Informatik and Medizintechnik.
- **The same kind of programme can be split over several types.** Competition is therefore counted by similarity of content in Step 3, not by membership in a type.
- **The catalogue's own subject labels did worse** (10 of 30) in the same kind of check. This suggests that the very short descriptions limit any grouping: half of the offers have only a title and a subject label.

### What we achieved in Step 2

We have a cleaned text for every offer and a list of 824 programme types that comes from the data, not from us. The method is repeatable, and independent reviewers recomputed its numbers and found no mismatch. We were honest about its limit: the first result failed our own quality check, we did not lower the bar or repair types by hand, and we changed three rules openly before any programme was judged. The remaining weakness (about 1 mixed type in 4) is handled by a check on every type in Step 3.

## Step 3: which programme types stay, and how much competition do they have?

**Question.** Which types are open to OTH and inside EMI's field? How many similar programmes exist near Amberg, in Bavaria, in the south and in Germany?

**What we did.** Every type was split into a bachelor and a master *unit* (where at least 2 universities offer it). Every unit got the same four tests:
1. **Already at OTH?** If OTH offers a programme of this kind at this level, the unit goes.
2. **Teacher training?** If half or more of its offers are teaching degrees, it goes (a faculty cannot open a state teaching qualification alone).
3. **Shrinking?** If its subject loses more than 3 % of German first-years (master: graduates) a year in Bavaria *and* this trend is reliable, it goes.
4. **Inside EMI's field?** We compare the titles and focus areas of its offers with the titles of the modules EMI teaches.

Then, for every unit, we counted similar programmes by content, not by name. Competition is only measured; it never removes a unit.

### Three rules had to be changed

The rules were written before we knew that Step 2 would give many small types. With small types three rules gave results that were plainly wrong. We changed them openly, each by a check that does not look at any candidate. **Soma confirmed these changes on 15 Nov 2026.**

| Rule | What went wrong | What we do now |
|---|---|---|
| Merging near-identical types | Types were linked through chains, so two giant groups (one for bachelor, one for master) held two thirds of all units (mechatronics together with Asian studies), and each whole group was removed as "already at OTH". Only 43 units were left after all four tests. | A group forms only if every pair in it is close. The largest group has 6 units. |
| Inside EMI's field | The first score put Informatik and 30 of 49 clearly EMI-related units outside EMI's field, and let through units such as Philosophie (2 Fächer) and Kunstgeschichte (2-Fächer). | An independent AI reader marked 300 units from their titles. We compared eight ways of measuring and took the best one. |
| Already at OTH | Half of OTH's own programmes did not trigger the test, so existing OTH programmes stayed on the candidate list. | A unit goes if an OTH programme belongs to its type (or lies very close to its centre). 45 of OTH's 46 programmes now trigger the test. |

In [14]:
S3 = json.loads((ROOT / "data" / "interim" / "funnel2_step3.json").read_text(encoding="utf-8"))
cal, dr = S3["calibration"], S3["dropped"]
show(pd.DataFrame([
    ("Units tested (type x bachelor/master)", f"{S3['units']:,}"),
    ("Removed: already at OTH", dr["already at OTH"]),
    ("Removed: teacher training", dr["teacher training"]),
    ("Removed: shrinking on a reliable trend", dr["falling validated trend"]),
    ("Removed: outside EMI's field", dr["outside EMI's field"]),
    ("Units that stay (primary list)", f"{S3['primary']} = {S3['primary_candidates']} candidates after merging near-identical ones"),
    ("How well the EMI-field score separates the 300 marked units (0.5 = chance, 1 = perfect)", f"{cal['auc']} (the first score: {cal['auc_of_the_score_as_first_written']})"),
    ("Marked EMI-field units that the bar keeps", f"{cal['field_units_kept']} of {cal['field_units']}"),
    ("Marked other units that the bar also keeps", f"{cal['outside_units_kept']} of {cal['outside_units']}"),
    ("Same result when run a second time", "yes"),
], columns=["", "Result"]))

,Result
Units tested (type x bachelor/master),"1,285"
Removed: already at OTH,44
Removed: teacher training,244
Removed: shrinking on a reliable trend,17
Removed: outside EMI's field,859
Units that stay (primary list),365 = 268 candidates after merging near-identical ones
"How well the EMI-field score separates the 300 marked units (0.5 = chance, 1 = perfect)",0.965 (the first score: 0.713)
Marked EMI-field units that the bar keeps,47 of 49
Marked other units that the bar also keeps,49 of 251
Same result when run a second time,yes


### The top of the list, and the competition counts

Sorted by the EMI-field score. "Similar offers" are programmes with similar content: Germany-wide and within 150 km of Amberg. The last column shows whether an independent reader found the type to be one kind of programme.

In [15]:
SC = pd.read_csv(ROOT / "data" / "interim" / "funnel2_screen.csv", float_precision="round_trip")
CO = pd.read_csv(ROOT / "docs" / "funnel_type_coherence.csv", keep_default_na=False)[["type", "mixed"]]
P = SC[SC["primary"]].merge(CO, on="type", how="left").sort_values("emi score", ascending=False)
P["one kind of programme?"] = P["mixed"].map({False: "yes", True: "mixed", "False": "yes", "True": "mixed"})
cols = {"type": "Type", "level": "Level", "central titles": "Typical titles", "emi score": "EMI score", "similar offers": "Similar offers in Germany",
        "similar within 150 km": "within 150 km", "one kind of programme?": "One kind of programme?"}
show(P.head(20).assign(**{"central titles": P["central titles"].str.slice(0, 90), "emi score": P["emi score"].round(2)})[list(cols)].rename(columns=cols))
fl = S3["primary_flags"]
show(pd.DataFrame([("No similar programme within 150 km of Amberg", fl["no similar offer within 150 km"]),
                   ("Exists elsewhere in Germany, but not in Bavaria", fl["absent in Bavaria"]),
                   ("Exists elsewhere, but not within 150 km and not in the south (the supervisor's tip)", fl["gap (none within 150 km and none in the south)"]),
                   ("Subject not shrinking in the Laender outside the south that offer it (whole list)", fl["popularity tier 2"])], columns=["Units on the list with ...", "Number"]))
ok = (CO[CO["type"].isin(P["type"])]["mixed"].astype(str) == "False")
print(f"Types on the list read by an independent reader: {len(ok)}; one kind of programme: {int(ok.sum())} ({ok.mean():.0%}; our bar was 80 %). The others carry the flag 'mixed'.")

Type,Level,Typical titles,EMI score,Similar offers in Germany,within 150 km,One kind of programme?
12,bachelor,"Physik; Physics; Physik / Physics; Mathematische Physik; Physik, Theoretische",0.890000,53,12,yes
438,master,Mechatronische Systeme; Mechatronische Systementwicklung; Mechatronische und cyberphysisch,0.840000,28,5,yes
12,master,"Physik; Physics; Physik / Physics; Mathematische Physik; Physik, Theoretische",0.790000,53,11,yes
206,bachelor,Elektrotechnik; Elektrotechnik / Electrical Engineering; Electrical Engineering; Elektrote,0.760000,283,41,yes
6,master,Mathematik; Mathematik (2-Fächer); Mathematik in Medizin und Lebenswissenschaften; Mathema,0.710000,65,10,yes
206,master,Elektrotechnik; Elektrotechnik / Electrical Engineering; Electrical Engineering; Elektrote,0.700000,193,39,yes
684,bachelor,Creative Technologies; Creative Engineering,0.690000,2,1,mixed
736,master,Elektrotechnik - Erneuerbare Energien und Systemtechnik (dt.-frz. Studiengang); Elektrotec,0.680000,6,0,yes
235,master,Embedded Systems; Embedded Systems Engineering; Embedded Systems - Aerospace Engineering;,0.670000,7,0,yes
6,bachelor,Mathematik; Mathematik (2-Fächer); Mathematik in Medizin und Lebenswissenschaften; Mathema,0.670000,78,12,yes


Units on the list with ...,Number
No similar programme within 150 km of Amberg,90
"Exists elsewhere in Germany, but not in Bavaria",80
"Exists elsewhere, but not within 150 km and not in the south (the supervisor's tip)",33
Subject not shrinking in the Laender outside the south that offer it (whole list),30


Types on the list read by an independent reader: 247; one kind of programme: 172 (70%; our bar was 80 %). The others carry the flag 'mixed'.


### What the result means

- **365 units stay, in 268 candidates.** The list is deliberately generous: by the reader's marks about half of it is still outside EMI's field. Physics and mathematics rank high because EMI teaches modules with these titles. Step 7 measures the real fit on curricula; a unit removed here would be lost for good.
- **About 3 types in 10 on the list are mixed** (70 % hold one kind of programme; our bar was 80 %). They keep the flag "mixed" and cannot become a candidate without a full reading of their titles.
- **The competition counts are not validated.** We tested whether "similar content" really means "the same kind of programme" on 100 offers, twice. By Soma's decision three AI readers marked the offers. Just inside our similarity line 46 % (second round 64 %) were the same kind; we needed 70 %. Just outside it 40 % (46 %) were the same kind; we allowed 30 %. So the line is not sharp. The counts show roughly how crowded a field is, not the number of direct competitors, and they get the widest uncertainty range in the final ranking.
- **The supervisor's tip is now a number:** 33 units exist elsewhere in Germany but neither within 150 km of Amberg nor in the south.

### What we achieved in Step 3

Every programme type got the same four tests, and every unit has its reason for staying or going. We found three of our own rules failing, showed why, and replaced them using checks that do not look at any candidate; independent reviewers recomputed the numbers of the final result for all units. We now have a list of 268 candidates with a first count of their competition. The three changed rules are confirmed by Soma. The competition counts stay in the analysis with the flag "not validated".

## Step 4: are students coming?

**Question.** How do the student numbers behind every candidate develop?

**What we did.** The statistics office counts students by subject, not by programme. So every unit gets a *pool*: the subjects of its offers, weighted by how often they occur. For each pool we looked at German first-years in Bavaria (for master units: graduates), the same for Germany, and foreign students separately. We also tested whether a trend helps to forecast: we pretended to stand in past years and compared three simple forecasts with what really happened.

In [16]:
S4 = json.loads((ROOT / "data" / "interim" / "funnel2_step4.json").read_text(encoding="utf-8"))
tb, tg, med, mase = S4["primary_by_trend_in_bavaria"], S4["primary_by_trend_in_germany"], S4["primary_median_trend"], S4["average_MASE"]
show(pd.DataFrame([
    ("Units with student numbers", f"{S4['units']:,} (on our list: {S4['primary']})"),
    ("On our list: German students growing in Bavaria", tb["growing (0 or more)"]),
    ("On our list: falling by up to 3 % a year", tb["falling slowly (0 to -3)"]),
    ("On our list: falling faster than 3 % a year", tb["falling fast (below -3)"]),
    ("On our list: German students growing in Germany", tg["growing (0 or more)"]),
    ("Middle unit of our list, German students, % a year", f"{med['German BY']} in Bavaria, {med['German DE']} in Germany"),
    ("Middle unit of our list, foreign students, % a year", f"+{med['foreign BY']} in Bavaria, +{med['foreign DE']} in Germany"),
    ("Forecast test, average error (lower is better)", f"'next year = this year' {mase['naive']}, straight trend {mase['drift']}, smoothing {mase['ets']}"),
    ("Units that pass our test for a reliable trend", f"{S4['units_validated']} of {S4['units']:,} ({S4['primary_validated']} on our list)"),
], columns=["", "Result"]))

,Result
Units with student numbers,"1,285 (on our list: 365)"
On our list: German students growing in Bavaria,128
On our list: falling by up to 3 % a year,93
On our list: falling faster than 3 % a year,136
On our list: German students growing in Germany,95
"Middle unit of our list, German students, % a year","-2.03 in Bavaria, -1.92 in Germany"
"Middle unit of our list, foreign students, % a year","+9.32 in Bavaria, +6.26 in Germany"
"Forecast test, average error (lower is better)","'next year = this year' 2.008, straight trend 2.409, smoothing 2.53"
Units that pass our test for a reliable trend,"51 of 1,285 (23 on our list)"


### The units with the strongest growing and falling German pools (pools of 100 students or more)

In [17]:
DM = pd.read_csv(ROOT / "data" / "interim" / "funnel2_demand.csv", float_precision="round_trip")
P4 = DM[DM["primary"]].copy()
P4["German students, last year"] = P4["German 2024 BY"].where(P4["level"] == "bachelor", P4["German 2025 BY"]).round(0)
big = P4[P4["German students, last year"] >= 100].sort_values("trend German BY %/yr", ascending=False)
cols = {"type": "Type", "level": "Level", "central titles": "Typical titles", "pool": "Subject pool", "German students, last year": "German students in the pool, last year",
        "trend German BY %/yr": "German, Bavaria, % a year", "trend German DE %/yr": "German, Germany, % a year", "trend foreign BY %/yr": "Foreign, Bavaria, % a year"}
fmt = lambda x: x.assign(**{"central titles": x["central titles"].str.slice(0, 60)})[list(cols)].rename(columns=cols).round(1)
show(fmt(big.head(10)))
show(fmt(big.tail(6)))

Type,Level,Typical titles,Subject pool,"German students in the pool, last year","German, Bavaria, % a year","German, Germany, % a year","Foreign, Bavaria, % a year"
245,master,Mediendesign; Medien - Mediendesign; Medienwissenschaft; Dig,SF287:1.00,222.000000,22.200000,5.400000,7.700000
408,master,Media Art and Design; Medienkunst/Mediengestaltung; Experime,SF287:1.00,222.000000,22.200000,5.400000,7.700000
662,master,Design Management; Designmanagement,SF287:0.75; SF007:0.25,166.000000,21.200000,4.500000,5.300000
603,bachelor,Bionik; Bionik dual; Materialwissenschaften und Bionik,SF072:0.83; SF294:0.17,435.000000,13.300000,-1.200000,15.700000
815,bachelor,Service Engineering dual; Service Engineering,SF072:1.00,520.000000,13.200000,-1.100000,15.700000
220,master,Informationstechnologie; Informatik; Information Technology;,SF222:0.93; SF079:0.04; SF123:0.04,158.000000,10.200000,1.400000,9.300000
180,master,Medienwissenschaften; Medienwissenschaft; Medienkulturwissen,SF302:0.96; SF149:0.01; SF155:0.01; SF303:0.01,430.000000,8.600000,0.800000,9.100000
268,master,Informatik - Software und System Engineering; Informatik: So,SF079:0.67; SF222:0.33,2175.000000,7.200000,5.600000,17.600000
149,master,Angewandte Informatik; Angewandte Informatik/Infotronik; Ang,SF079:0.71; SF221:0.14; SF312:0.14,2335.000000,7.200000,5.800000,18.400000
517,master,Informatik - Vertrauenswürdige Systeme; Informatik (dt.-frz.,SF079:1.00,3246.000000,7.100000,5.800000,18.300000


Type,Level,Typical titles,Subject pool,"German students in the pool, last year","German, Bavaria, % a year","German, Germany, % a year","Foreign, Bavaria, % a year"
176,bachelor,Kommunikations- und Medienwissenschaft; Medien- und Kommunik,SF303:0.84; SF302:0.16,286.000000,-8.800000,-2.900000,13.900000
512,bachelor,User Experience Design; Usability Engineering,SF121:0.83; SF287:0.17,107.000000,-8.900000,-6.400000,4.300000
237,bachelor,Energietechnik; Energie- und Umwelttechnik: Energietechnik;,SF211:0.49; SF213:0.30; SF226:0.06; SF123:0.05,158.000000,-9.400000,-5.600000,15.700000
507,bachelor,Visual Computing; Computervisualistik; Computervisualistik u,SF121:1.00,108.000000,-10.200000,-6.500000,4.200000
443,bachelor,Medienproduktion; Medieningenieur; Media: Conception & Produ,SF305:0.75; SF302:0.25,175.000000,-10.200000,-8.100000,5.600000
502,bachelor,Tonmeister; Tonmeister*in für audiovisuelle Medien,SF305:1.00,151.000000,-12.700000,-9.700000,5.000000


### What the result means

- **Most candidates sit in subjects that lose German students.** On our list 229 of 365 units have a falling German pool in Bavaria; the middle unit loses about 2 % a year. Foreign students grow in most pools. A new programme has to win students in a shrinking German market or attract international students.
- **Trends are weak evidence.** On average the simple forecast "next year is like this year" beats a trend line. Only 51 of 1,285 units pass our test for a reliable trend. A slope tells us what happened since 2019, not what will happen.
- **The numbers describe subjects, not programmes.** A niche programme inside "Informatik" shares the trend of all of Informatik. That is the best the public statistics allow.

### What we achieved in Step 4

Every candidate now has the student numbers of its subjects: German and foreign, Bavaria and Germany, with an honest statement of how much the trend can be trusted. No candidate was removed in this step. We found that student trends alone cannot separate the candidates well, so the next steps (jobs, expected first-years, fit with EMI) carry more weight.

## Step 5: are there jobs?

**Question.** How many job ads match each candidate, in Germany, in Bavaria and within 100 km of Amberg?

**What we did.** In the first analysis we chose the search words ourselves. Now a fixed rule makes them: for every type we take the words that most of its programme titles share (at most three). The list of search words was saved and committed before we asked the job board of the Federal Employment Agency for any count. All counts of our list are from one day.

**An error we caught.** The job board read "Amberg" as a village of the same name near Buchloe in Swabia, 177 km away. An independent reviewer found it in the stored replies. We now send "Amberg, Oberpfalz", check the place in every reply, and counted everything again. The same error is in the job numbers of our first analysis ("191 ads within 100 km"), which must be corrected there.

In [18]:
S5 = json.loads((ROOT / "data" / "interim" / "funnel2_step5.json").read_text(encoding="utf-8"))
a, p5 = S5["all_ads"], S5["primary"]
show(pd.DataFrame([
    ("Units on our list with all counts", f"{p5['labour input ok']} of {p5['units']}"),
    ("All job ads on that day", f"{a['germany']:,} in Germany, {a['bayern']:,} in Bavaria, {a['amberg_100km']:,} within 100 km of Amberg"),
    ("Middle unit of our list: ads within 100 km for its middle search word", int(p5["median of the measure"])),
    ("Units with no ad within 100 km", p5["units with 0 ads within 100 km"]),
    ("Units with fewer than 10 ads", p5["units below 10"]),
    ("Units with 150 ads or more", p5["units with 150 or more"]),
    ("Bavaria's share of a unit's German ads (middle unit; all ads: {:.1%})".format(S5["bavaria_share_of_all_ads"]), f"{p5['median Bavarian share']:.1%}"),
], columns=["", "Result"]))

,Result
Units on our list with all counts,365 of 365
All job ads on that day,"780,392 in Germany, 131,440 in Bavaria, 49,476 within 100 km of Amberg"
Middle unit of our list: ads within 100 km for its middle search word,22
Units with no ad within 100 km,63
Units with fewer than 10 ads,152
Units with 150 ads or more,82
Bavaria's share of a unit's German ads (middle unit; all ads: 16.8%),20.2%


### The units with the most ads, and some specific ones

In [19]:
LB = pd.read_csv(ROOT / "data" / "interim" / "funnel2_labour.csv")
P5 = LB[LB["primary"]].sort_values("ads within 100 km (median keyword)", ascending=False)
cols = {"type": "Type", "level": "Level", "central titles": "Typical titles", "keywords": "Search words (made by rule)",
        "ads within 100 km (median keyword)": "Ads within 100 km (middle search word)", "ads in Germany (sum of keywords)": "Ads in Germany (all search words)"}
fmt = lambda x: x.assign(**{"central titles": x["central titles"].str.slice(0, 60)})[list(cols)].rename(columns=cols)
show(fmt(P5.drop_duplicates("type").head(10)))
show(fmt(P5[P5["type"].isin([221, 235, 283, 438, 648, 793])].drop_duplicates("type")))

Type,Level,Typical titles,Search words (made by rule),Ads within 100 km (middle search word),Ads in Germany (all search words)
55,master,Mechatronik; Elektrotechnik/Mechatronik; Mechatronik - Allge,mechatronik,2254.000000,30774.000000
487,master,Führung und Organisation; Führung in Dienstleistungsunterneh,führung; organisation,2190.500000,71667.000000
413,master,Elektrotechnik und Energiesysteme; Elektrische Systeme; Ener,elektrotechnik,2009.000000,27339.000000
463,master,"Gebäudetechnik; Gebäude-, Energie- und Klimatechnik; Gebäude",-informationstechnik; energie-; gebäude-,1685.000000,811905.000000
331,master,Elektrotechnik; Elektrotechnik und Informationstechnik; Elek,elektrotechnik; informationstechnik; elektro-,1587.000000,50392.000000
226,bachelor,Elektrotechnik und Informationstechnik; Elektro- und Informa,informationstechnik; elektrotechnik; elektro-,1587.000000,50392.000000
237,master,Energietechnik; Energie- und Umwelttechnik: Energietechnik;,energietechnik; energie-; gebäudetechnik,1483.000000,48515.000000
96,master,Maschinenbau; Allgemeiner Maschinenbau; Maschinenbau (berufs,maschinenbau,1256.000000,19266.000000
669,bachelor,Kunstwissenschaften; Kunst und Kommunikation; Kunst und Kuns,kunst,1175.000000,13085.000000
457,master,Management and Technology; Business Innovation & Technology;,management; technology,1135.000000,37153.000000


Type,Level,Typical titles,Search words (made by rule),Ads within 100 km (middle search word),Ads in Germany (all search words)
221,bachelor,Robotik und Automatisierung; Autonome Systeme; Automation an,robotik; robotics; automation,85.000000,3617.000000
438,bachelor,Mechatronische Systeme; Mechatronische Systementwicklung; Me,mechatronische systeme,73.000000,870.000000
283,bachelor,Data Science; Computer Science; Data Science und Künstliche,data science; artificial intelligence; künstliche intelligenz,67.000000,2372.000000
235,bachelor,Embedded Systems; Embedded Systems Engineering; Embedded Sys,embedded systems engineering,22.000000,250.000000
793,master,Artificial Intelligence; Artificial Intelligence and Intelli,artificial intelligence,6.000000,169.000000
648,bachelor,Künstliche Intelligenz in der Medizin; Künstliche Intelligen,künstliche intelligenz in der medizin,1.000000,25.000000


### What the result means

- **The count tells us how often a word appears in job ads, not how many jobs wait for graduates of a programme.** General words lead the list: Mechatronik, Organisation, Management, Elektrotechnik. That says these words are common, not that a new programme with this name is needed.
- **Specific programmes get small, more honest counts:** data science 109, Robotik 85, mechatronic systems 73, Künstliche Intelligenz 67, embedded systems engineering 22 ads within 100 km.
- **The measure separates only the middle.** 82 units are above the top of our scale (150 ads) and 152 are below its bottom (10 ads).
- **Some search words are weak.** The rule keeps word stems such as "elektro-". It allowed striking such words by hand; we did no hand work, so they stay and are named in the notes.
- **Jobs are a little more Bavarian than average:** for the middle unit 20 % of the German ads are in Bavaria (all ads: 17 %).

### What we achieved in Step 5

Every candidate has job-ad counts from search words that a rule made, not we. We found and corrected a place error that also affects our first analysis. We learned that this measure is rough: it is useful to see whether a field has any jobs nearby, but it cannot rank similar candidates finely. No candidate was removed.

## Next: Step 6 (first-years new to OTH)

- For every candidate: how many first-years could a programme in Amberg or Weiden expect, given where school leavers live and where similar programmes already are?
- How many of them would be new to OTH, and how many would come from OTH's existing programmes?